<a href="https://colab.research.google.com/github/MiguelAP6/IAAC/blob/Pedro-Ferreira/Data_Preparation_PF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# Criar a pasta 'src' se não existir
os.makedirs("src", exist_ok=True)

# Escrever o módulo Python reutilizável
code_src = """import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

def load_and_clean_data(df_input, target_col='Threat Level'):
    '''Limpa valores em falta e remove colunas irrelevantes (IDs e Datas).'''
    df = df_input.dropna().copy()

    # Remover colunas de identificadores/datas
    id_cols = [c for c in df.columns if 'id' in c.lower() or 'date' in c.lower() or 'timestamp' in c.lower()]
    if id_cols:
        df = df.drop(columns=id_cols)

    return df

def prepare_pipeline(df, target_col='Threat Level'):
    '''Executa LabelEncoding, StandardScaler e Train-Test Split estratificado.'''
    if target_col not in df.columns:
        target_col = df.columns[-1]

    X = df.drop(columns=[target_col])
    y = df[target_col]

    label_encoders = {}
    for col in X.select_dtypes(include=['object', 'category']).columns:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        label_encoders[col] = le

    target_encoder = LabelEncoder()
    y_encoded = target_encoder.fit_transform(y.astype(str))

    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.20, random_state=42, stratify=y_encoded
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return {
        'X_train_scaled': X_train_scaled,
        'X_test_scaled': X_test_scaled,
        'y_train': y_train,
        'y_test': y_test,
        'scaler': scaler,
        'target_encoder': target_encoder,
        'label_encoders': label_encoders,
        'feature_names': list(X.columns)
    }
"""

with open("src/data_processing.py", "w") as f:
    f.write(code_src)

print("Módulo 'src/data_processing.py' criado com sucesso na pasta 'src/'!")

Módulo 'src/data_processing.py' criado com sucesso na pasta 'src/'!


In [ ]:
import pandas as pd
from google.colab import files
from src.data_processing import load_and_clean_data, prepare_pipeline

# 1. Carregar o dataset caso a variável 'df' não esteja na memória deste caderno
if 'df' not in globals():
    print("Dataset não encontrado na memória. Por favor, faz o upload do ficheiro CSV:")
    uploaded = files.upload()
    file_name = list(uploaded.keys())[0]
    df = pd.read_csv(file_name)

# 2. Execução da preparação através do módulo src/data_processing.py
df_clean = load_and_clean_data(df)
prep_data = prepare_pipeline(df_clean)

# 3. Extração dos conjuntos prontos para modelação
X_train_scaled = prep_data['X_train_scaled']
X_test_scaled = prep_data['X_test_scaled']
y_train = prep_data['y_train']
y_test = prep_data['y_test']
scaler = prep_data['scaler']
target_encoder = prep_data['target_encoder']
label_encoders = prep_data['label_encoders']
feature_names = prep_data['feature_names']

print("\n=== PARTE 2: RESUMO DA PREPARAÇÃO DOS DADOS (M) ===")
print(f"Conjunto de Treino: {X_train_scaled.shape[0]} amostras")
print(f"Conjunto de Teste: {X_test_scaled.shape[0]} amostras")
print(f"Número de Atributos (Features): {X_train_scaled.shape[1]}")
print(f"Classes do Alvo: {list(target_encoder.classes_)}")

Dataset não encontrado na memória. Por favor, faz o upload do ficheiro CSV:


Saving archive.zip to archive.zip

=== PARTE 2: RESUMO DA PREPARAÇÃO DOS DADOS (M) ===
Conjunto de Treino: 2400 amostras
Conjunto de Teste: 600 amostras
Número de Atributos (Features): 8
Classes do Alvo: ['AI-based Detection', 'Antivirus', 'Encryption', 'Firewall', 'VPN']
